# 01 Data prep

**Plan section:** 2 (data) and 11 (reproducibility). **Label:** supporting.

Converts the raw CSV into `data/processed/criteo.parquet` (float32 features, int8 flags, original row order) and runs structural checks: column names, row count, nulls, binary flags, finite features.

No outcomes are compared between arms here. That happens only in `03_ate`, after the validity checks in `02_validity`.

In [1]:
import pandas as pd

from uplift.config import PARQUET, RAW_CSV
from uplift.data import csv_to_parquet, load_processed, validate_parquet
from uplift.io import write_results

In [2]:
if not PARQUET.exists() or PARQUET.stat().st_mtime < RAW_CSV.stat().st_mtime:
    csv_to_parquet()

report = validate_parquet()
write_results("data_prep", report)
print(f"rows: {report['n_rows']:,}   parquet on disk: {report['file_size_mb']} MB")

rows: 13,979,592   parquet on disk: 112.1 MB


In [3]:
pd.DataFrame(
    {
        "dtype": report["dtypes"],
        "nulls": report["nulls"],
        "non_binary": report["non_binary_flags"],
        "non_finite": report["non_finite_features"],
    }
).loc[report["columns"]].astype(object).fillna("-")

,dtype,nulls,non_binary,non_finite
f0,FLOAT,0,-,0.0
f1,FLOAT,0,-,0.0
f2,FLOAT,0,-,0.0
f3,FLOAT,0,-,0.0
f4,FLOAT,0,-,0.0
f5,FLOAT,0,-,0.0
f6,FLOAT,0,-,0.0
f7,FLOAT,0,-,0.0
f8,FLOAT,0,-,0.0
f9,FLOAT,0,-,0.0


In [4]:
df = load_processed()
print(f"in-memory size: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.head()

in-memory size: 0.73 GB


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616364,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616364,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616364,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616364,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616364,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
